# Food Recognition — image classification (gated) + ingredient calorie regression

Two-part pipeline:
1. **Image -> Food-101 class** (fine-tune MobileNetV3/ViT). Requires the local
   `data/food101/images` (DVC) or the HF `ethz/food101` fallback — the cell
   checks availability and degrades gracefully.
2. **Ingredients -> calories** (runs fully local on the 231k Food.com recipes)
   and a Food-101 class->median-calories lookup table for explainability.
Exports: `food_calorie_regressor` ONNX for meal-plan calorie estimation.

In [1]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'datasets', 'pandas', 'matplotlib'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth, on_gpu, tf_version
set_memory_growth()
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


2026-08-04 15:45:17.889569: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-04 15:45:18.012890: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-08-04 15:45:20.537785: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TF 2.20.0 | GPU: False | scale: smoke


2026-08-04 15:45:23.586912: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Part 1a — check Food-101 image availability (local DVC, or tiny opt-in HF sample)
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import buddy_data as D

FOOD_IMAGE_OK = False
food_root = None
try:
    food_root = D.require('food101')
    print('local Food-101 images:', food_root)
    FOOD_IMAGE_OK = True
except FileNotFoundError:
    print('no local Food-101 images (dvc pull data/food101). '
          'Set BUDDY_FOOD_IMAGES=1 for a tiny ~180-image HF sample instead.')

if not FOOD_IMAGE_OK and os.environ.get('BUDDY_FOOD_IMAGES') == '1':
    try:
        from datasets import load_dataset
        ds = load_dataset('ethz/food101', split='train', streaming=True)
        out = Path('../data/processed/food101_hf_subset')
        labels = ds.features['label'].names[:6]
        counts = {}
        for ex in ds:
            name = labels[ex['label']]
            (out / name).mkdir(parents=True, exist_ok=True)
            cnt = counts.get(name, 0)
            if cnt < 30:
                ex['image'].save(out / name / f'{cnt}.jpg')
                counts[name] = cnt + 1
            if sum(counts.values()) >= 180:
                break
        if sum(counts.values()) >= 60:
            FOOD_IMAGE_OK, food_root = True, out
        print('HF subset ready:', counts)
    except Exception as e:
        print('HF unavailable:', type(e).__name__, str(e)[:120])
print('FOOD_IMAGE_OK =', FOOD_IMAGE_OK)

no local Food-101 images (dvc pull data/food101). Set BUDDY_FOOD_IMAGES=1 for a tiny ~180-image HF sample instead.
FOOD_IMAGE_OK = False


In [3]:
# Part 1b — image fine-tune (gated; runs only when images are present)
if FOOD_IMAGE_OK:
    IMG = 224
    train_dir = food_root / 'images' / 'train'      # canonical Food-101 layout
    if not train_dir.exists():
        train_dir = food_root / 'train'
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=(IMG, IMG), batch_size=32, shuffle=True)
    n_classes = len(train_ds.class_names)
    train_ds = train_ds.map(lambda x, y: (x / 255.0, y)).prefetch(tf.data.AUTOTUNE)
    if SCALE == 'smoke':
        train_ds = train_ds.take(6)

    base = tf.keras.applications.MobileNetV3Small(weights='imagenet', include_top=False,
                                                  input_shape=(IMG, IMG, 3))
    base.trainable = False
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    img_model = tf.keras.Model(base.input, out)
    img_model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy',
                      metrics=['accuracy'])
    img_model.fit(train_ds, epochs={'smoke': 1, 'demo': 3, 'full': 6}[SCALE], verbose=1)
    onnx = export_keras_onnx(img_model, Path('../models'), 'food_classifier', '1.0.0')
    print('exported food_classifier:', onnx)
else:
    print('image branch skipped (no Food-101 images); continue with calorie regression below')

image branch skipped (no Food-101 images); continue with calorie regression below


In [4]:
# Part 2 — ingredient -> calorie regression (fully local, 231k recipes)
from buddy_data import food_com_recipes
import ast

N = {'smoke': 20_000, 'demo': 120_000, 'full': 231_637}[SCALE]
rec = food_com_recipes()
rng = np.random.default_rng(3)
rec = rec.sample(n=min(N, len(rec)), random_state=3)

rec['ing_text'] = rec['ingredients_list'].map(lambda xs: ' '.join(xs).lower())
rec['calories'] = rec['calories'].clip(0, 5000)
rec['logcal'] = np.log1p(rec['calories']).astype(np.float32)
print('recipes:', rec.shape, '| calorie stats (kcal):')
print(rec['calories'].describe(percentiles=[.25, .5, .75, .95]).round(0).to_string())

recipes: (20000, 22) | calorie stats (kcal):
count    20000.0
mean       462.0
std        592.0
min          0.0
25%        175.0
50%        314.0
75%        522.0
95%       1271.0
max       5000.0


In [5]:
# Text vectorizer + small regression head over ingredients
MAX_TOKENS, SEQ, EMB = 40_000, 128, 64
vec = tf.keras.layers.TextVectorization(max_tokens=MAX_TOKENS, output_sequence_length=SEQ,
                                        standardize='lower_and_strip_punctuation')
vec.adapt(np.array(rec['ing_text']))
print('ingredient vocab size:', vec.vocabulary_size())

Xin = vec(np.array(rec['ing_text'])).numpy()
yin = rec['logcal'].to_numpy()
split = int(0.8 * len(rec))
inp = tf.keras.Input(shape=(SEQ,), dtype='int64')
x = tf.keras.layers.Embedding(MAX_TOKENS + 2, EMB)(inp)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
out = tf.keras.layers.Dense(1, activation='linear')(x)
cal_m = tf.keras.Model(inp, out)
cal_m.compile(tf.keras.optimizers.Adam(1e-3), 'mse')
cal_m.fit(Xin[:split], yin[:split], epochs={'smoke': 1, 'demo': 3, 'full': 6}[SCALE],
          batch_size=256, validation_data=(Xin[split:], yin[split:]), verbose=1)

ingredient vocab size: 2736


 1/63 ━━━━━━━━━━━━━━━━━━━━ 1:40 2s/step - loss: 35.2725

 2/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 32.2958

 3/63 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - loss: 29.4607

 5/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 24.1261

...

58/63 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 3.8135

59/63 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 3.7690

61/63 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 3.6834

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 3.6198

63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - loss: 3.6198 - val_loss: 1.2049


In [6]:
# Evaluate calorie regression in original kcal (MAE/median abs error)
from sklearn.metrics import mean_absolute_error, median_absolute_error

pred_log = cal_m.predict(Xin[split:], batch_size=512)[:, 0]
mae = float(mean_absolute_error(np.expm1(yin[split:]), np.expm1(pred_log)))
mdae = float(median_absolute_error(np.expm1(yin[split:]), np.expm1(pred_log)))
print(f'calorie MAE={mae:.0f} kcal | median AE={mdae:.0f} kcal')

# class -> median calories lookup (Food-101 explainability)
from food101_labels import FOOD101_LABELS
name = rec['name'].str.lower()
lookup = {}
for cls in list(FOOD101_LABELS)[:5]:
    key = cls.replace('_', ' ')
    mask = name.str.contains(key)
    if mask.sum():
        lookup[cls] = float(rec.loc[mask, 'calories'].median())
print('sample class->median kcal:', lookup)

1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step

7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step 

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


calorie MAE=319 kcal | median AE=186 kcal
sample class->median kcal: {'apple_pie': 524.9, 'baklava': 401.15000000000003, 'beignets': 581.6, 'bread_pudding': 341.7, 'cannoli': 354.5}


### Export contract (consumed by the AI service)

The cells below write `../models/food_calorie_regressor.onnx` and its dynamic-INT8 quantized copy
`food_calorie_regressor_int8.onnx`. `app/ml/serving.py::load_preferred('food_calorie_regressor')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [7]:
# Export calorie regressor ONNX (+ INT8) + vectorizer vocab
import json
from pathlib import Path

onnx = export_keras_onnx(cal_m, Path('../models'), 'food_calorie_regressor', '1.0.0',
                         input_signature=[tf.TensorSpec((None, SEQ), tf.int64, name='input_ids')])
q = quantize_dynamic_onnx(onnx)
(Path('../models') / 'food_calorie_vectorizer.json').write_text(
    json.dumps({'max_tokens': MAX_TOKENS, 'sequence_length': SEQ,
                'vocabulary': vec.get_vocabulary()}))
mlflow_log({'name': 'food_calorie_regressor', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'cal_mae_kcal': round(mae, 1), 'cal_median_ae_kcal': round(mdae, 1)}})
print('exported', q)

Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a warning and probably safe to ignore.
'NoneType' object has no attribute 'name'


I0000 00:00:1785847555.270791  873893 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785847555.270973  873893 single_machine.cc:376] Starting new session


I0000 00:00:1785847555.542292  873893 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785847555.542639  873893 single_machine.cc:376] Starting new session


{
  "name": "food_calorie_regressor",
  "version": "1.0.0",
  "artifact_path": "../models/food_calorie_regressor-1.0.0_int8.onnx",
  "framework": "tensorflow",
  "metrics": {
    "cal_mae_kcal": 318.8,
    "cal_median_ae_kcal": 186.0
  }
}
exported ../models/food_calorie_regressor-1.0.0_int8.onnx
